In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub

print("Installation terminée ! Connexion à Hugging Face...")

# connexion à Hugging Face
from huggingface_hub import notebook_login
notebook_login()

In [2]:
from datasets import load_dataset

dataset = load_dataset('mlabonne/guanaco-llama2-1k', split='train')
dataset = dataset.train_test_split(test_size=0.1) # on divise le dataset en données d'enrainement et de test

train_dataset = dataset['train']
test_dataset = dataset['test']

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Le nom exact du modèle que nous allons fine-tuné
model_name = "meta-llama/Llama-2-7b-hf"

# Configuration de la compression (Quantification 4-bit)
#le but ici c'est de réduire la taille du model pour pouvoir économiser en VRAM et l'éxécuter sur le GPU (j'utilise le GPU T4 de colab)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # le format 4bit adapté pour les réseaux de neurones
    bnb_4bit_compute_dtype=torch.float16, # au moment de l'entrainement on laisse les se faire en calcul 16 bit pour augmenter la présicion des poids et diminuer le temps d'entrainement
    bnb_4bit_use_double_quant=False,
)

# Chargement du tokenizer adapté a notre model
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # On utilise le token de fin de phrase (end of sentence) comme token de remplissage pour les cas ou les phrases n'ont pas la meme taille
tokenizer.padding_side = "right" # On ajoute le eos_token à droite du texte pour le remplissage

print("Configuration et Tokenizer chargés avec succès !")

In [ ]:
print("Téléchargement du modèle en cours...")
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map={"": 0})
model.config.use_cache = False
print("Modèle chargé avec succès sur le GPU !")

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    lora_alpha=16, # ce parametres determine l'influence du calque loRA (de l'exemple du principe de la photo transparente que j'ai trouvé sur internet!)
    # sur le model d'origine! en gros çà permet de choisir a quelle point on veut que le model considere ou adopte les connaissances acquises lors du fine tuning (souvent au detriment de ses connaissances d'origine)
    lora_dropout=0.1, # On désactive au hasard 10% des poids pour éviter que les matrices additionnelles de LoRA entraine du surapprentissage
    r=64, # Le Rang définit la complexité des matrices de notre adaptateur (calque) LoRA. Plus il est grand, plus le modèle peut apprendre des détails complexes, mais çà augmente aussi le temps de calcul et la mémoire requise pour l'entrainement.
    bias="none",
    task_type="CAUSAL_LM", # le Type de tâche! ici c'est la Génération de texte
)

print("Configuration LoRA prête !")

In [ ]:
from trl import SFTConfig
from transformers import EarlyStoppingCallback

training_arguments = SFTConfig(
    output_dir="./results", # on sauvegarde les poids du model ici
    per_device_train_batch_size=4, # un batch de 4 signifie que le model s'entraine sur 4 exemples simultanément
    gradient_accumulation_steps=4, # les gradients sont stocké temporairement pour plus tard mettre à jour les poids apres un bloc de 16 exemples (4 batch x 4 steps équvalent à 1 step)
    optim="paged_adamw_32bit", # un optimiseur spécial qui gère bien la mémoire
    save_steps=25, # sauvegarder l'avancer de l'entrainerment apres 25 epochs
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001, # pénalité sur les poids! comme lasso!
    fp16=False, # nous avons déjà géré les calcul temporaires en 16 bits lors de la configuration BitsAndBytes pus haut!
    bf16=False, # bf16 pas compatible sur ce GPU
    max_grad_norm=0.3, # clipping de gradient
    max_steps=400, # ici on limite le test à 400 entrée, pas sur l'esemble du dataset! le but ici c'est de voir rapidement le model changer de comporter pour verifer si le finetuning à réussi!
    warmup_steps=3,
    lr_scheduler_type="constant",
    dataset_text_field = "text", # on spécifie la colonne ou se trouve les exemple à traiter par le model,
    eval_strategy="steps",        # On évalue le modèle à chaque étape de sauvegarde
    eval_steps=25,                # Tous les 25 pas, on vérifie la loss de validation
    load_best_model_at_end=True,  # On garde le meilleur modèle
    metric_for_best_model="loss",
    greater_is_better=False
)

early_stop = EarlyStoppingCallback(early_stopping_patience=2)

print("Paramètres d'entraînement définis !")

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset= test_dataset,
    processing_class = tokenizer,
    peft_config = peft_config,
    args = training_arguments,
    callbacks=[early_stop]
)
print("Lancement de l'entraînement...")
trainer.train()
print("Fin de l'entraînement !")

In [18]:
# L'entrainement est terminé, c'est l'heure de vérité!!!
# Le nom du dossier où on va sauvegarder notre adaptateur
new_model_name = "llama-2-7b-finetune"

# On sauvegarde le model et le tokenizer
trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)

print("Modèle et tokenizer sauvegardés localement !")

Modèle et tokenizer sauvegardés localement !


In [ ]:
from transformers import pipeline

# le prompt de test
prompt = " what is supervised learning?"

pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=200,
    temperature=0.7,          # ces qautres derniers parametres sont fixé pour donné plus de cohérence et de diversité à l'output de notre model finetuné
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True
)

print("Génération en cours...\n")

result = pipe(prompt)

print(result[0]['generated_text'])

In [ ]:
# Push du model finetuné vers hugging face
model_hub_path = "JauresN16/llama-2-7b-custom-finetune"

trainer.model.push_to_hub(model_hub_path)
tokenizer.push_to_hub(model_hub_path)

print(f"model dispo sur : https://huggingface.co/{model_hub_path}")